# DADA-2000 original — Phase 6 / **T2 learning curve** (step B)

Plan: `.project/plans/katvad-t2-learning-curve.md` — **§5 (pre-registration) was written before this
notebook ran.** Authorized 2026-09-26: fixed **2,040 steps** at every fraction (D-1), **T2 micro**
primary with the macro sign (D-2), fractions **25 % + 50 %** (D-3). The 100 % point is phase 4's
KIP-off arm, **reused, not re-run**.

**The question:** does halving T2's train *source videos* cost AUC at fixed compute? It decides CCD:

| verdict | condition (Δ50 = AUC(100 %) − AUC(50 %), paired by seed, n = 3) | next |
|---|---|---|
| **RISING** | Δ50 micro t95 **lower > 0** and Δ50 macro point > 0 | write the CCD plan (G-dup, G-pos) |
| **FLAT** | Δ50 micro t95 **upper < +0.010** | close CCD → write-up (D) |
| **INCONCLUSIVE** | anything else | CCD stays parked → D. **No seed extension.** |

Everything is **KIP-off** (phase 5: KIP-v1 is neutral on T2). Nothing in `core/` is edited here.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE = '/content/drive/MyDrive/Thesis'
os.environ['PROJECT_ROOT']       = DRIVE
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE}/outputs'
for v in ('KATVAD_DATA_ROOT', 'KATVAD_CACHE_ROOT', 'KATVAD_CKPT_ROOT', 'KATVAD_OUTPUT_ROOT'):
    os.makedirs(os.environ[v], exist_ok=True)

os.environ['REPO'] = f'{DRIVE}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'tools' / 'subset_train.py').is_file(), (
    f'{REPO} has no core/tools/subset_train.py -- sync core/ from the repo first')
os.environ['PYTHONPATH'] = str(REPO)
sys.path.insert(0, str(REPO))

from core import constants  # noqa: E402
from core.config import load_config  # noqa: E402

DATASET = constants.DADA_ORIGIN_DATASET
T2 = constants.DATA_ROOT / DATASET
CLIP_DIR = constants.CLIP_CACHE_DIR / DATASET
DOTA_DATA = constants.DATA_ROOT / 'DoTA' / 'labels_s8'
DOTA_CLIP = constants.CLIP_CACHE_DIR / 'DoTA_s8_ncc'
P4_REPORTS = constants.OUTPUT_ROOT / 'REPORTS' / f'{DATASET}_phase4'   # the 100 % arm
RUNS = constants.OUTPUT_ROOT / f'{DATASET}_lcurve'                    # Drive, per arm
REPORTS = constants.OUTPUT_ROOT / 'REPORTS' / f'{DATASET}_lcurve'
P6 = Path('/content/p6')                                              # VM-local scratch
P6.mkdir(parents=True, exist_ok=True)

# --- the pre-registered design (plan 3, 5) -----------------------------------
SEEDS = (2024, 2025, 2026)          # = phase 4's KIP-off seeds (D-7)
FRACTIONS = (0.5, 0.25)             # 25 % is NESTED in 50 % per seed (D-3, D-4)
TOTAL_STEPS = 2040                  # phase 4: 20 epochs x 102 steps (D-1)
STEP_TOLERANCE = 0.02               # G-S4
KERNEL = constants.DADA_ORIGIN_SCORE_HEAD_KERNEL   # 3
TOPK_PCT = constants.DADA_ORIGIN_MIL_TOPK_PCT      # 5
BATCH = load_config().train.batch_size             # 64, the same default phase 4 used
FLAT_UPPER = 0.010                  # FLAT: Delta50 micro t95 upper < +0.010
COLLAPSE_GAP = 0.02                 # macro < micro by more than this -> C14 flag
T95_N3 = 4.302652729911275
ORACLE_T2 = 0.7037
CHUNK_STEPS = 510                   # ~5 phase-4 epochs per invocation

for name, path in (('corpus', T2), ('clip', CLIP_DIR), ('phase-4 reports', P4_REPORTS),
                   ('DoTA data', DOTA_DATA), ('DoTA clip', DOTA_CLIP)):
    print(f'  {name:16s} {path}   {"OK" if path.exists() else "MISSING"}')
print(f'batch {BATCH} | kernel {KERNEL} | mil_topk_pct {TOPK_PCT} | {TOTAL_STEPS} steps per arm')

In [ ]:
%%bash
# Same pins as phase_4.ipynb (transformers internals, lesson C7).
pip install -q "transformers==4.56.*" av einops faiss-cpu
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU'

## 0. Stage the caches on local disk

`core/train.py` indexes the dataset serially in the main process; reading features over Drive FUSE
is most of the wall clock. Bit-identical copies, C2 untouched.

In [ ]:
import json
import shutil
import subprocess
import time

STAGE = P6 / 'stage'
STAGE.mkdir(parents=True, exist_ok=True)
ENV = dict(os.environ)


def stage(src, name):
    if src is None or not src.exists():
        return src
    dst = STAGE / name
    if dst.exists():
        print(f'  {name:12s} already staged')
        return dst
    t0 = time.time()
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    files = [f for f in dst.rglob('*') if f.is_file()] if dst.is_dir() else [dst]
    print(f'  {name:12s} {len(files):5d} files, '
          f'{sum(f.stat().st_size for f in files) / 2**20:7.1f} MiB in {time.time() - t0:5.1f}s')
    return dst


def run(cmd, capture=False):
    proc = subprocess.run([str(c) for c in cmd], cwd=str(REPO), env=ENV,
                          capture_output=capture, text=True)
    if proc.returncode:
        if capture:
            print(proc.stdout or '', proc.stderr or '', sep='\n')
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd)}')
    return proc.stdout if capture else None


L_CLIP = stage(CLIP_DIR, 'clip')
L_T2 = stage(T2, 'corpus')
L_DOTA_CLIP = stage(DOTA_CLIP, 'dota_clip')
L_DOTA_DATA = stage(DOTA_DATA, 'dota_data')

windows = json.loads((L_T2 / constants.WINDOWS_FILENAME).read_text())
assert {w['end'] - w['start'] for w in windows.values()} == {constants.DADA_ORIGIN_WINDOW_LENGTH}, \
    'the corpus on disk is not the W=20 build Gate W passed'

## 1. Build the subsets — gates G-S1 … G-S3, G-S6

`core.tools.subset_train` draws whole **source videos**, stratified by accident type, and **raises**
instead of writing a subset that leaks a test source (G-S1), loses a class (G-S2) or escapes its
`--nest-in` parent (G-S3). The draw is deterministic per `(seed, fraction)`; a rebuild after a
disconnect must reproduce the `kept_sources_sha1` already recorded on Drive.

In [ ]:
SUBSETS = P6 / 'subsets'


def tag(frac, seed):
    return f'f{round(frac * 100):03d}_s{seed}'


SUB = {}
for seed in SEEDS:
    parent = None
    for frac in FRACTIONS:                      # 0.5 first, then 0.25 nested in it
        out = SUBSETS / tag(frac, seed)
        if not (out / constants.SUBSET_MANIFEST_FILENAME).exists():
            cmd = [sys.executable, '-m', 'core.tools.subset_train', '--data-dir', L_T2,
                   '--out-dir', out, '--fraction', frac, '--seed', seed]
            if parent is not None:
                cmd += ['--nest-in', parent]
            run(cmd)
        manifest = json.loads((out / constants.SUBSET_MANIFEST_FILENAME).read_text())
        recorded = REPORTS / 'subsets' / f'{tag(frac, seed)}.json'
        if recorded.exists():
            assert json.loads(recorded.read_text())['kept_sources_sha1'] == \
                manifest['kept_sources_sha1'], f'{tag(frac, seed)} rebuilt DIFFERENTLY'
        SUB[(frac, seed)] = (out, manifest)
        parent = out

full = json.loads((L_T2 / constants.LABELS_TRAIN_FILENAME).read_text())
print(f'100 %: {len(full)} train windows, {sum(full.values())} abnormal')
print(f'{"subset":>12} {"sources":>9} {"windows":>9} {"abnormal":>9} {"item frac":>9}  dropped types')
for (frac, seed), (out, m) in SUB.items():
    gs6 = abs(m['item_fraction'] - frac) <= 0.03
    print(f'{tag(frac, seed):>12} {m["sources_kept"]:>4}/{m["sources_total"]:<4} '
          f'{m["items_kept"]:>9} {m["abnormal_kept"]:>9} {m["item_fraction"]:>9.3f}'
          f'{"" if gs6 else " (G-S6: off by > 0.03 -- reported)"}  {m["dropped_types"] or "-"}')

## 2. One KNN cache per subset — gate G-S5

The DVS neighbour pool is built from `labels_train.json`. Reusing the full corpus's cache would
splice in normal windows the subset arm never trains on.

In [ ]:
import numpy as np

KNN = {}
for key, (out, m) in SUB.items():
    knn = out / constants.KNN_CACHE_FILENAME
    if not knn.exists():
        run([sys.executable, '-m', 'core.data.knn_cache', '--data-dir', out,
             '--dataset', DATASET, '--clip-dir', L_CLIP, '--output', knn])
    labels = json.loads((out / constants.LABELS_TRAIN_FILENAME).read_text())
    with np.load(knn, allow_pickle=True) as z:
        ids = set(z['anomaly_ids'].tolist())
        pool = set(z['neighbor_ids'].ravel().tolist())
    assert ids == {i for i, v in labels.items() if v == 1}, f'G-S5 FAIL {tag(*key)}'
    assert pool <= {i for i, v in labels.items() if v == 0}, f'{tag(*key)}: foreign neighbours'
    KNN[key] = knn
print(f'G-S5 PASS on {len(KNN)} caches')

## 3. Fixed compute — gate G-S4

`steps/epoch = ceil(2 · N_abnormal / batch)` (`core/train.py`), and `train.num_epochs` also sets the
cosine horizon, so the epoch count is derived per subset to land on 2,040 steps.

In [ ]:
import math

EPOCHS = {}
for key, (out, m) in SUB.items():
    spe = math.ceil(2 * m['abnormal_kept'] / BATCH)
    epochs = round(TOTAL_STEPS / spe)
    steps = spe * epochs
    assert abs(steps - TOTAL_STEPS) / TOTAL_STEPS <= STEP_TOLERANCE, \
        f'G-S4 FAIL {tag(*key)}: {steps} steps'
    EPOCHS[key] = (epochs, spe, steps)
    print(f'{tag(*key):>12}: {spe:3d} steps/epoch x {epochs:3d} epochs = {steps} steps')

## 4. Train — stage 2, KIP-off, in chunks synced to Drive

Same command line as phase 4's KIP-off arm; only `--data-dir`, `--knn-cache` and `train.num_epochs`
change. §6 asserts that on the written configs.

In [ ]:
def arm_dir(key):
    return RUNS / tag(*key)


def local_dir(drive_dir):
    return P6 / 'runs' / drive_dir.relative_to(RUNS)


def epochs_done(run_dir):
    path = run_dir / 'metrics.jsonl'
    if not path.exists():
        return 0
    last = -1
    for line in path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            last = max(last, json.loads(line)['epoch'])
    return last + 1


def sync(src, dst):
    dst.mkdir(parents=True, exist_ok=True)
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        if (src / name).exists() and not (src == dst):
            shutil.copy2(src / name, dst / name)


def train_arm(key):
    out, _ = SUB[key]
    epochs, spe, _ = EPOCHS[key]
    chunk = max(1, round(CHUNK_STEPS / spe))
    drive_out, local = arm_dir(key), local_dir(arm_dir(key))
    if drive_out.exists() and not local.exists():
        sync(drive_out, local)
    while (done := epochs_done(local)) < epochs:
        seed = key[1]
        cmd = [sys.executable, '-m', 'core.train',
               '--set', 'train.stage=2', '--set', f'train.seed={seed}',
               '--set', f'train.num_epochs={epochs}', '--set', 'train.amp=true',
               '--set', f'data.dataset={DATASET}',
               '--set', f'model.score_head_kernel={KERNEL}',
               '--set', f'loss.mil_topk_pct={TOPK_PCT}',
               '--set', 'kip.enabled=false',
               '--data-dir', out, '--clip-dir', L_CLIP, '--knn-cache', KNN[key],
               '--output-dir', local,
               '--stop-after-epochs', min(chunk, epochs - done)]
        if done:
            cmd += ['--resume', local / 'checkpoint_last.pt']
        print(f'  {tag(*key)}: epochs {done} -> {min(done + chunk, epochs)} of {epochs}')
        run(cmd)
        sync(local, drive_out)
    print(f'  {tag(*key)}: {epochs_done(local)}/{epochs} epochs -- done')


for key in SUB:
    train_arm(key)

## 5. Evaluate — in-domain T2 and zero-shot DoTA

Same test sets as phase 4 (the subset dirs carry the parent's `frame_labels_test.json` byte for
byte). `--score-norm auto` resolves exactly as it did there.

In [ ]:
def evaluate(ckpt, out_dir, *, dataset, data_dir, clip_dir):
    if (out_dir / 'results.json').exists():
        return json.loads((out_dir / 'results.json').read_text())
    run([sys.executable, '-m', 'core.evaluate', '--ckpt', ckpt,
         '--set', f'data.dataset={dataset}', '--data-dir', data_dir, '--clip-dir', clip_dir,
         '--output-dir', out_dir, '--save-scores'])
    return json.loads((out_dir / 'results.json').read_text())


EVALS = {}
for key in SUB:
    ckpt = local_dir(arm_dir(key)) / 'checkpoint_last.pt'
    if not ckpt.exists():
        ckpt = arm_dir(key) / 'checkpoint_last.pt'
    EVALS[(key, 't2')] = evaluate(ckpt, arm_dir(key) / 'eval_t2', dataset=DATASET,
                                  data_dir=L_T2, clip_dir=L_CLIP)
    EVALS[(key, 'dota')] = evaluate(ckpt, arm_dir(key) / 'eval_dota', dataset='DoTA',
                                    data_dir=L_DOTA_DATA, clip_dir=L_DOTA_CLIP)
print(f'{len(EVALS)} evaluations')

## 6. Pairing check — the subset arms differ from phase 4's KIP-off arm in `num_epochs` only

`--data-dir` and `--knn-cache` are CLI paths, not config keys (C17), so the config diff must be
exactly `train.num_epochs`.

In [ ]:
import yaml


def flat(d, prefix=''):
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out.update(flat(v, f'{prefix}{k}.'))
        else:
            out[f'{prefix}{k}'] = v
    return out


for key in SUB:
    mine = flat(yaml.safe_load((arm_dir(key) / 'config.yaml').read_text()))
    ref = flat(yaml.safe_load((P4_REPORTS / f'config_s{key[1]}_kip_off.yaml').read_text()))
    diff = sorted(k for k in set(mine) | set(ref) if mine.get(k) != ref.get(k))
    assert diff == ['train.num_epochs'], f'{tag(*key)} differs from phase 4 in {diff}'
print('pairing OK: every subset arm differs from its phase-4 KIP-off seed in train.num_epochs only')

## 7. Read-out — the pre-registered table (plan §5.2)

In [ ]:
import statistics


def full_res(seed, bench):
    return json.loads((P4_REPORTS / f'results_s{seed}_kip_off_{bench}.json').read_text())


def value(frac, seed, bench, metric):
    res = full_res(seed, bench) if frac == 1.0 else EVALS[((frac, seed), bench)]
    return float(res[metric])


def paired(hi, lo, bench, metric):
    d = [value(hi, s, bench, metric) - value(lo, s, bench, metric) for s in SEEDS]
    m = statistics.mean(d)
    h = T95_N3 * statistics.stdev(d) / len(d) ** 0.5
    return m, m - h, m + h, ' '.join('+' if x > 0 else '-' for x in d)


print(f'{"fraction":>8} | {"T2 micro":>9} {"T2 macro":>9} | {"DoTA micro":>10} {"DoTA macro":>10}')
COLLAPSED = []
for frac in (1.0, 0.5, 0.25):
    row = {(b, m): statistics.mean(value(frac, s, b, m) for s in SEEDS)
           for b in ('t2', 'dota') for m in ('auc', 'auc_macro')}
    print(f'{frac:>8.2f} | {row["t2", "auc"]:>9.4f} {row["t2", "auc_macro"]:>9.4f} | '
          f'{row["dota", "auc"]:>10.4f} {row["dota", "auc_macro"]:>10.4f}')
    for s in SEEDS:
        mic, mac = value(frac, s, 't2', 'auc'), value(frac, s, 't2', 'auc_macro')
        if mac < mic - COLLAPSE_GAP:
            COLLAPSED.append((frac, s, mic, mac))
print(f'T2 micro is read beside the clip oracle {ORACLE_T2}')
if COLLAPSED:
    print('C14 FLAG (macro < micro - 0.02), excluded from the verdict:', COLLAPSED)

print()
ROWS = {}
for name, hi, lo in (('Delta50', 1.0, 0.5), ('Delta25', 0.5, 0.25)):
    for bench in ('t2', 'dota'):
        for metric in ('auc', 'auc_macro'):
            ROWS[(name, bench, metric)] = r = paired(hi, lo, bench, metric)
            print(f'{name} {bench:>4} {metric:>9}: {r[0]:+.4f}  t95 [{r[1]:+.4f}, {r[2]:+.4f}]  {r[3]}'
                  + ('   <- PRIMARY' if (name, bench, metric) == ('Delta50', 't2', 'auc') else ''))

mic = ROWS[('Delta50', 't2', 'auc')]
mac = ROWS[('Delta50', 't2', 'auc_macro')]
excluded = any(f in (1.0, 0.5) for f, *_ in COLLAPSED)
if excluded:
    VERDICT = 'INCONCLUSIVE (C14 flag on a Delta50 arm)'
elif mic[1] > 0 and mac[0] > 0:
    VERDICT = 'RISING'
elif mic[2] < FLAT_UPPER:
    VERDICT = 'FLAT'
else:
    VERDICT = 'INCONCLUSIVE'
NEXT = {'RISING': 'data is a bottleneck -> write the CCD plan (G-dup vs DoTA, then G-pos)',
        'FLAT': 'close CCD -> write-up (D)'}.get(VERDICT, 'CCD stays parked -> D; no seed extension')
print(f'\nVERDICT: {VERDICT}  ->  {NEXT}')
d50, d25 = mic[0], ROWS[('Delta25', 't2', 'auc')][0]
print(f'shape (reported, not decided on): Delta25 {d25:+.4f} vs Delta50 {d50:+.4f} -> '
      + ('diminishing returns' if d25 > d50 else 'not diminishing'))

## 8. Record the campaign (C17)

`outputs/` is gitignored: `REPORTS/DADA2000_orig_lcurve/` is what comes back to the repo.

In [ ]:
import hashlib

REPORTS.mkdir(parents=True, exist_ok=True)
(REPORTS / 'subsets').mkdir(exist_ok=True)
for key, (out, m) in SUB.items():
    (REPORTS / 'subsets' / f'{tag(*key)}.json').write_text(json.dumps(m, indent=2))
    shutil.copy2(arm_dir(key) / 'config.yaml', REPORTS / f'config_{tag(*key)}.yaml')
for (key, bench), res in EVALS.items():
    (REPORTS / f'results_{tag(*key)}_{bench}.json').write_text(json.dumps(res, indent=2))

digest = hashlib.sha256()
for path in sorted((REPO / 'core').rglob('*.py')):
    digest.update(path.relative_to(REPO).as_posix().encode())
    digest.update(path.read_bytes())

manifest = {
    'notebook': 'colab/DADA2000Origin/phase_6_learning_curve.ipynb',
    'plan': '.project/plans/katvad-t2-learning-curve.md',
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'core_sha256': digest.hexdigest()[:16],
    'design': {'seeds': list(SEEDS), 'fractions': list(FRACTIONS), 'total_steps': TOTAL_STEPS,
               'batch': BATCH, 'kernel': KERNEL, 'mil_topk_pct': TOPK_PCT, 'kip': 'off',
               'full_arm': str(P4_REPORTS)},
    'arms': {tag(*k): {'epochs': EPOCHS[k][0], 'steps_per_epoch': EPOCHS[k][1],
                       'steps': EPOCHS[k][2], 'kept_sources_sha1': SUB[k][1]['kept_sources_sha1'],
                       't2_auc': EVALS[(k, 't2')]['auc'],
                       't2_auc_macro': EVALS[(k, 't2')]['auc_macro'],
                       'dota_auc': EVALS[(k, 'dota')]['auc'],
                       'dota_auc_macro': EVALS[(k, 'dota')]['auc_macro']} for k in SUB},
    'rows': {f'{n} {b} {m}': {'delta': r[0], 't95': [r[1], r[2]], 'signs': r[3]}
             for (n, b, m), r in ROWS.items()},
    'c14_flags': COLLAPSED,
    'verdict': VERDICT,
    'next': NEXT,
}
(REPORTS / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
print('recorded ->', REPORTS)
for p in sorted(REPORTS.rglob('*')):
    if p.is_file():
        print(f'   {p.relative_to(REPORTS)}  ({p.stat().st_size / 1024:.1f} KiB)')